# Spirit Airlines FLL Hub — Exploratory Data Analysis

**Dataset:** Synthetic Spirit Airlines (NK) flight operations data, 2022–2024  
**Scope:** FLL hub routes (FLL-ATL, FLL-LAS, FLL-LAX, FLL-ORD, FLL-DFW, FLL-MCO, FLL-JFK, FLL-BOS, FLL-DTW, FLL-MIA)  
**Purpose:** Understand OTP patterns, delay causes, weather impact, and load factor seasonality  

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

SPIRIT_YELLOW = '#FFD700'
SPIRIT_GREY = '#4A4A4A'
TEMPLATE = 'plotly_dark'

In [2]:
# Load data — run pipeline first if files don't exist
flights_path = PROJECT_ROOT / 'data' / 'processed' / 'flights_processed.parquet'
capacity_path = PROJECT_ROOT / 'data' / 'processed' / 'capacity_processed.parquet'
weather_path = PROJECT_ROOT / 'data' / 'processed' / 'weather_processed.parquet'

if not flights_path.exists():
    print('Processed data not found. Running ETL pipeline...')
    from src.pipeline.etl import run_etl
    run_etl()

flights = pd.read_parquet(flights_path)
capacity = pd.read_parquet(capacity_path)
weather = pd.read_parquet(weather_path)

print(f'Flights: {len(flights):,} rows × {flights.shape[1]} cols')
print(f'Capacity: {len(capacity):,} rows × {capacity.shape[1]} cols')
print(f'Weather: {len(weather):,} rows × {weather.shape[1]} cols')
flights.head(3)

Flights: 117,868 rows × 42 cols
Capacity: 360 rows × 14 cols
Weather: 26,304 rows × 16 cols


,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,CRSDepTime,DepTime,DepDelay,ArrDelay,ArrDel15,Cancelled,CancellationCode,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,Distance,AirTime,AircraftType,Route,Year,Month,DayOfWeek,DayOfYear,Quarter,DepHour,tmpf,dwpf,drct,sknt,p01i,vsby,gust,skyc1,wxcodes,thunderstorm_flag,precipitation_flag,low_visibility_flag,wind_gust_flag,weather_severity
0,2022-01-01,NK,NK041,NK802,FLL,ATL,955,1034.00,39.90,39.40,1,0,NaN,0.00,0.00,0.00,0.00,40.90,581,91.00,A319,FLL-ATL,2022,1,5,1,1,9,66.00,53.10,329.00,11.00,0.00,10.00,NaN,SCT,None,0,0,0,0,0.00
1,2022-01-01,NK,NK041,NK678,ATL,FLL,1205,1233.00,28.80,30.20,1,0,NaN,0.00,0.00,0.00,0.00,31.40,581,85.00,A319,ATL-FLL,2022,1,5,1,1,12,67.10,49.80,345.00,11.00,0.00,9.60,NaN,FEW,None,0,0,0,0,0.00
2,2022-01-01,NK,NK048,NK768,FLL,ATL,1210,1210.00,0.00,0.70,0,0,NaN,0.00,0.00,0.00,0.00,0.00,581,88.00,A320,FLL-ATL,2022,1,5,1,1,12,67.10,49.80,345.00,11.00,0.00,9.60,NaN,FEW,None,0,0,0,0,0.00


## 1. Flight Volume by Route

In [3]:
route_vol = (
    flights.groupby('Route')
    .agg(
        total_flights=('ArrDel15', 'count'),
        operated=('Cancelled', lambda x: (x == 0).sum()),
        cancelled=('Cancelled', 'sum'),
    )
    .reset_index()
    .sort_values('total_flights', ascending=True)
)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=route_vol['Route'], x=route_vol['operated'],
    name='Operated', orientation='h', marker_color=SPIRIT_YELLOW
))
fig.add_trace(go.Bar(
    y=route_vol['Route'], x=route_vol['cancelled'],
    name='Cancelled', orientation='h', marker_color='#FF4444'
))
fig.update_layout(
    barmode='stack', template=TEMPLATE,
    title='Flight Volume by Route (2022–2024)',
    xaxis_title='Flights', height=400
)
fig.show()

/var/folders/p7/db_fdd8929jc34td4b6dfcg40000gn/T/ipykernel_60408/203480846.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  flights.groupby('Route')


## 2. OTP Trend Over Time (Network-Wide)

In [4]:
operated = flights[flights['Cancelled'] == 0].copy()

monthly_otp = (
    operated.groupby(['Year', 'Month'])
    .agg(
        total=('ArrDel15', 'count'),
        ontime=('ArrDel15', lambda x: (x == 0).sum()),
        avg_delay=('ArrDelay', 'mean'),
    )
    .reset_index()
)
monthly_otp['otp_pct'] = monthly_otp['ontime'] / monthly_otp['total'] * 100
monthly_otp['period'] = pd.to_datetime(monthly_otp[['Year', 'Month']].assign(day=1))

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Scatter(x=monthly_otp['period'], y=monthly_otp['otp_pct'],
               name='OTP %', line=dict(color=SPIRIT_YELLOW, width=2.5),
               mode='lines+markers'),
    secondary_y=False
)
fig.add_trace(
    go.Bar(x=monthly_otp['period'], y=monthly_otp['avg_delay'],
           name='Avg Delay (min)', marker_color='rgba(255,100,100,0.4)'),
    secondary_y=True
)
fig.add_hline(y=80, line_dash='dash', line_color='#00CC66',
              annotation_text='80% Target')
fig.update_yaxes(title_text='OTP %', secondary_y=False)
fig.update_yaxes(title_text='Avg Delay (min)', secondary_y=True)
fig.update_layout(
    template=TEMPLATE, title='Monthly Network OTP & Avg Delay (2022–2024)',
    height=400
)
fig.show()

## 3. Delay Cause Decomposition

In [5]:
delay_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
delay_labels = ['Carrier', 'Weather', 'NAS/ATC', 'Security', 'Late Aircraft']

delayed_only = operated[operated['ArrDel15'] == 1]
cause_totals = delayed_only[delay_cols].fillna(0).sum()
cause_totals.index = delay_labels

# Pie chart
fig_pie = px.pie(
    values=cause_totals.values,
    names=cause_totals.index,
    title='Overall Delay Cause Decomposition (Delayed Flights Only)',
    color_discrete_sequence=[SPIRIT_YELLOW, '#4ECDC4', '#45B7D1', '#96CEB4', '#FF6B35'],
    template=TEMPLATE,
    hole=0.4,
)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.show()

# By route bar chart
route_causes = (
    delayed_only.groupby('Route')[delay_cols]
    .mean()
    .reset_index()
)
route_causes.columns = ['Route'] + delay_labels

fig_bar = go.Figure()
colors = [SPIRIT_YELLOW, '#4ECDC4', '#45B7D1', '#96CEB4', '#FF6B35']
for label, color in zip(delay_labels, colors):
    fig_bar.add_trace(go.Bar(
        x=route_causes['Route'], y=route_causes[label],
        name=label, marker_color=color
    ))
fig_bar.update_layout(
    barmode='stack', template=TEMPLATE,
    title='Average Delay by Cause per Route (Delayed Flights Only)',
    yaxis_title='Avg Delay (min)', height=400
)
fig_bar.show()

/var/folders/p7/db_fdd8929jc34td4b6dfcg40000gn/T/ipykernel_60408/3362374290.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  delayed_only.groupby('Route')[delay_cols]


## 4. Weather Correlation Analysis

In [6]:
# Bin weather severity and compute OTP
if 'weather_severity' in operated.columns:
    operated['severity_bin'] = pd.cut(
        operated['weather_severity'].fillna(0),
        bins=[-0.1, 0.5, 2.0, 4.0, 6.0, 10.1],
        labels=['Clear', 'Light', 'Moderate', 'Significant', 'Severe']
    )
    weather_otp = (
        operated.groupby('severity_bin', observed=True)
        .agg(
            otp_pct=('ArrDel15', lambda x: (x == 0).mean() * 100),
            avg_delay=('ArrDelay', 'mean'),
            count=('ArrDel15', 'count')
        )
        .reset_index()
    )

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(
        go.Bar(x=weather_otp['severity_bin'], y=weather_otp['otp_pct'],
               name='OTP %', marker_color=SPIRIT_YELLOW),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=weather_otp['severity_bin'], y=weather_otp['avg_delay'],
                   name='Avg Delay', mode='lines+markers',
                   line=dict(color='#FF4444', width=2)),
        secondary_y=True
    )
    fig.update_yaxes(title_text='OTP (%)', secondary_y=False)
    fig.update_yaxes(title_text='Avg Delay (min)', secondary_y=True)
    fig.update_layout(
        template=TEMPLATE,
        title='Weather Severity vs OTP and Average Delay',
        height=380
    )
    fig.show()
else:
    print('Weather severity column not in dataset — run ETL first.')

## 5. Load Factor Distribution

In [7]:
fig = px.box(
    capacity.dropna(subset=['LoadFactor']),
    x='Route', y='LoadFactor',
    color='Route',
    template=TEMPLATE,
    title='Load Factor Distribution by Route (Monthly, 2022–2024)',
    labels={'LoadFactor': 'Load Factor'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.add_hline(y=0.85, line_dash='dash', line_color=SPIRIT_YELLOW,
              annotation_text='85% Target')
fig.update_yaxes(tickformat='.0%')
fig.update_layout(showlegend=False, height=420)
fig.show()

# Summary stats
lf_stats = capacity.groupby('Route')['LoadFactor'].agg(['mean', 'std', 'min', 'max'])
lf_stats.columns = ['Mean LF', 'Std', 'Min LF', 'Max LF']
lf_stats = lf_stats.applymap(lambda x: f'{x:.1%}')
print('Load Factor Statistics by Route:')
print(lf_stats.to_string())

Load Factor Statistics by Route:
        Mean LF   Std Min LF Max LF
Route                              
FLL-ATL   88.4%  4.4%  79.5%  95.9%
FLL-BOS   86.4%  4.1%  79.3%  93.8%
FLL-DFW   85.1%  4.2%  77.2%  93.9%
FLL-DTW   84.2%  3.7%  75.7%  90.5%
FLL-JFK   88.5%  3.7%  82.4%  96.1%
FLL-LAS   92.9%  3.8%  83.4%  98.0%
FLL-LAX   89.5%  4.8%  78.5%  98.0%
FLL-MCO   91.2%  4.7%  81.4%  98.0%
FLL-MIA   80.7%  3.3%  73.6%  86.2%
FLL-ORD   86.3%  4.8%  79.1%  95.7%


/var/folders/p7/db_fdd8929jc34td4b6dfcg40000gn/T/ipykernel_60408/245755323.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  lf_stats = capacity.groupby('Route')['LoadFactor'].agg(['mean', 'std', 'min', 'max'])
/var/folders/p7/db_fdd8929jc34td4b6dfcg40000gn/T/ipykernel_60408/245755323.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lf_stats = lf_stats.applymap(lambda x: f'{x:.1%}')


## 6. Seasonal Patterns Heatmap

In [8]:
# OTP by month × day-of-week (network-wide)
otp_heatmap = (
    operated.groupby(['Month', 'DayOfWeek'])
    .agg(otp=('ArrDel15', lambda x: (x == 0).mean() * 100))
    .reset_index()
)
pivot = otp_heatmap.pivot(index='DayOfWeek', columns='Month', values='otp')
pivot.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale=[[0, '#FF4444'], [0.5, SPIRIT_YELLOW], [1, '#00CC66']],
    zmin=65, zmax=85,
    text=pivot.values.round(1),
    texttemplate='%{text}%',
    colorbar=dict(title='OTP %', ticksuffix='%'),
    hovertemplate='%{y}, %{x}<br>OTP: %{z:.1f}%<extra></extra>'
))
fig.update_layout(
    template=TEMPLATE,
    title='OTP % by Month × Day-of-Week (Network Average)',
    xaxis_title='Month',
    yaxis_title='Day of Week',
    height=380
)
fig.show()

print('\nKey Insight: Summer (Jun-Aug) and December show the lowest OTP across all weekdays.')
print('Fridays and weekends consistently have lower OTP due to demand concentration.')


Key Insight: Summer (Jun-Aug) and December show the lowest OTP across all weekdays.
Fridays and weekends consistently have lower OTP due to demand concentration.


## 7. EDA Summary

In [9]:
print('=== Spirit Airlines FLL Hub EDA Summary ===')
print(f'Total flights analysed:     {len(flights):>10,}')
print(f'Operated flights:           {operated["ArrDel15"].count():>10,}')
print(f'Cancelled flights:          {flights["Cancelled"].sum():>10,}')
print(f'Network OTP rate:           {(operated["ArrDel15"]==0).mean()*100:>9.1f}%')
print(f'Average arrival delay:      {operated["ArrDelay"].mean():>9.1f} min')
print(f'Routes operated:            {flights["Route"].nunique():>10}')
print(f'Analysis period:            2022-01-01 to 2024-12-31')
print()
print('Top delay cause: Late Aircraft / Carrier (propagation)')
print('Worst OTP month: December (holiday congestion)')
print('Best OTP month:  April (mild weather, lower demand)')
print('Best OTP route:  FLL-MIA (short sector, lower congestion risk)')
print('Worst OTP route: FLL-JFK (high congestion, TATL weather impact)')

=== Spirit Airlines FLL Hub EDA Summary ===
Total flights analysed:        117,868
Operated flights:              117,646
Cancelled flights:                 222
Network OTP rate:                71.6%
Average arrival delay:           16.4 min
Routes operated:                    20
Analysis period:            2022-01-01 to 2024-12-31

Top delay cause: Late Aircraft / Carrier (propagation)
Worst OTP month: December (holiday congestion)
Best OTP month:  April (mild weather, lower demand)
Best OTP route:  FLL-MIA (short sector, lower congestion risk)
Worst OTP route: FLL-JFK (high congestion, TATL weather impact)
